In [ ]:
!pip install geemap earthengine-api geopandas -q

import ee
import geemap
import numpy as np

ee.Authenticate()
ee.Initialize(project='flood-risk-mapping-500013')  # use your own GEE-linked Cloud project ID

In [ ]:
assam = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Assam'))
aoi = assam.geometry()

Map2 = geemap.Map(basemap='HYBRID')
Map2.centerObject(aoi, 7)
Map2.addLayer(aoi, {'color': 'red'}, 'AOI - Assam')
Map2

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").clip(aoi)

Map3 = geemap.Map(basemap='HYBRID')
Map3.centerObject(aoi, 7)
Map3.addLayer(dem, {'min': 0, 'max': 200,
                     'palette': ['darkblue', 'blue', 'lightblue', 'white']},
              'DEM')
Map3.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map3

In [ ]:
slope = ee.Terrain.slope(dem)

Map3b = geemap.Map(basemap='HYBRID')
Map3b.centerObject(aoi, 7)
Map3b.addLayer(slope, {'min': 0, 'max': 20, 'palette': ['white', 'orange', 'red']}, 'Slope')
Map3b.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map3b

In [ ]:
hillshade = ee.Terrain.hillshade(dem)

Map3c = geemap.Map(basemap='HYBRID')
Map3c.centerObject(aoi, 7)
Map3c.addLayer(hillshade, {}, 'Hillshade')
Map3c.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map3c

In [ ]:
flow_acc = ee.Image("WWF/HydroSHEDS/15ACC").clip(aoi)

Map4 = geemap.Map(basemap='HYBRID')
Map4.centerObject(aoi, 7)
Map4.addLayer(flow_acc, {'min': 0, 'max': 5000, 'palette': ['white', 'blue']}, 'Flow Accumulation')
Map4.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map4

In [ ]:
rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
    .filterDate('2022-06-01', '2022-07-15') \
    .filterBounds(aoi) \
    .sum() \
    .clip(aoi)

Map5 = geemap.Map(basemap='HYBRID')
Map5.centerObject(aoi, 7)
Map5.addLayer(rain, {'min': 0, 'max': 800, 'palette': ['white', 'blue', 'darkblue']}, 'Rainfall (mm)')
Map5.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map5

In [ ]:
low_elevation = dem.lt(60)
heavy_rain = rain.gt(400)
high_flow_acc = flow_acc.gt(2000)

flood_risk = low_elevation.add(heavy_rain).add(high_flow_acc)

aoi_mask = ee.Image.constant(1).clip(aoi).mask()
flood_risk_clipped = flood_risk.updateMask(aoi_mask)

Map6 = geemap.Map(basemap='HYBRID')
Map6.centerObject(aoi, 7)
Map6.addLayer(flood_risk_clipped,
              {'min': 0, 'max': 3, 'palette': ['white', 'lightblue', 'blue', 'darkblue']},
              'Flood Risk Classes')
Map6.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map6

In [ ]:
risk_category = flood_risk_clipped.expression(
    "(b(0) == 0) ? 1" +
    " : (b(0) == 1) ? 2" +
    " : 3"
).rename('Risk_Category').updateMask(aoi_mask)

Map7 = geemap.Map(basemap='HYBRID')
Map7.centerObject(aoi, 7)
Map7.addLayer(risk_category,
              {'min': 1, 'max': 3, 'palette': ['lightblue', 'blue', 'darkblue']},
              'Flood Risk Category')
Map7.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map7

In [ ]:
risk_vectors = risk_category.reduceToVectors(
    geometry=aoi,
    scale=200,
    geometryType='polygon',
    labelProperty='risk_class',
    reducer=ee.Reducer.countEvery(),
    maxPixels=1e10
)

Map8 = geemap.Map(basemap='HYBRID')
Map8.centerObject(aoi, 7)
Map8.addLayer(risk_vectors, {}, 'Flood Risk Vectors')
Map8.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map8

In [ ]:
pixel_area = ee.Image.pixelArea()

for cls in [1, 2, 3]:
    class_mask = risk_category.eq(cls)
    area_img = pixel_area.updateMask(class_mask)
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=200,
        maxPixels=1e10
    )
    area_sqkm = ee.Number(stats.get('area')).divide(1e6).getInfo()
    print(f"Risk class {cls}: {area_sqkm:.2f} sq.km")

In [ ]:
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filterDate('2022-06-15', '2022-07-15') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .select('VV') \
    .median() \
    .clip(aoi)

Map10 = geemap.Map(basemap='HYBRID')
Map10.centerObject(aoi, 7)
Map10.addLayer(s1, {'min': -25, 'max': 0}, 'Sentinel-1 SAR (VV)')
Map10.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam')
Map10

In [ ]:
task1 = ee.batch.Export.image.toDrive(
    image=risk_category,
    description='Brahmaputra_Flood_Risk_Map',
    folder='FloodRiskProject_Brahmaputra',
    scale=100,
    region=aoi,
    maxPixels=1e10
)
task1.start()

task2 = ee.batch.Export.table.toDrive(
    collection=risk_vectors,
    description='Brahmaputra_Flood_Risk_Vectors',
    folder='FloodRiskProject_Brahmaputra',
    fileFormat='SHP'
)
task2.start()

print("Exports started — check the Tasks tab in GEE Console or your Google Drive shortly.")

In [ ]:
Map_final = geemap.Map(basemap='HYBRID')
Map_final.centerObject(aoi, 7)
Map_final.addLayer(dem, {'min': 0, 'max': 200,
                          'palette': ['darkblue', 'blue', 'lightblue', 'white']},
                    'DEM', shown=False)
Map_final.addLayer(rain, {'min': 0, 'max': 800, 'palette': ['white', 'blue', 'darkblue']},
                    'Rainfall (mm)', shown=False)
Map_final.addLayer(risk_category,
                    {'min': 1, 'max': 3, 'palette': ['lightblue', 'blue', 'darkblue']},
                    'Flood Risk Category', shown=True)
Map_final.addLayer(s1, {'min': -25, 'max': 0}, 'Sentinel-1 SAR (VV)', shown=False)
Map_final.addLayer(aoi, {'color': 'blue'}, 'AOI - Assam', shown=True)
Map_final.addLayerControl()
Map_final.save('brahmaputra_flood_risk_map.html')

from google.colab import files
files.download('brahmaputra_flood_risk_map.html')

Map_final

In [ ]:
from google.colab import drive
drive.mount('/content/drive')